# Stage 9 — Improve Answer Format

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Package each response into a clean structure — `answer`, `relevant_papers`, `topics` — ready for direct use in the Streamlit UI (Stage 11).

**Before running:**
1. Upload `faiss_index.zip` (from Stage 6).
2. Make sure your `GROQ_API_KEY` Colab Secret is still set up (from Stage 8).

## Cell 1 — Install packages

In [1]:
!pip install -q langchain-groq faiss-cpu langchain-community langchain-huggingface sentence-transformers langchain-core pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Cell 2 — Rebuild the pipeline from Stage 8

Same setup as Stage 8: load the API key, reload the vector store, create the retriever, LLM, and prompt.

In [2]:
import os
import zipfile
from google.colab import userdata
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

with zipfile.ZipFile("faiss_index.zip", "r") as zip_ref:
    zip_ref.extractall("faiss_index")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

SYSTEM_PROMPT = """You are ResearchMate, a research paper assistant.
Answer the user's question using ONLY the research paper context provided below.

Rules:
- Do not invent or assume any information that is not present in the context.
- If the context does not contain enough information to answer the question, say so clearly instead of guessing.
- When you use information from a paper, mention its title.
- Keep your answer clear and concise.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(f"Title: {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

answer_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Pipeline ready.")

/tmp/ipykernel_5384/776311675.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pipeline ready.


## Cell 3 — Define the structured response shape (Pydantic model)

`RAGResponse` defines exactly what every answer from our chatbot will contain: the generated `answer` text, the list of `relevant_papers` (titles), and the list of `topics` covered. This is our "Structured Output" concept — a typed, predictable shape the Streamlit UI can rely on.

In [3]:
from pydantic import BaseModel
from typing import List

class RAGResponse(BaseModel):
    answer: str
    relevant_papers: List[str]
    topics: List[str]

print("RAGResponse schema defined.")

RAGResponse schema defined.


## Cell 4 — Build the combined `ask_researchmate` function

This function:
1. Retrieves the relevant documents directly (so we have reliable metadata).
2. Generates the natural-language answer via the LLM chain.
3. Extracts paper titles and a deduplicated, sorted list of topics straight from the retrieved documents' metadata — no LLM involved in this part, so it's always accurate.
4. Packages everything into a `RAGResponse`.

In [4]:
def ask_researchmate(question):
    docs = retriever.invoke(question)

    answer_text = answer_chain.invoke(question)

    relevant_papers = [doc.metadata["title"] for doc in docs]

    all_topics = set()
    for doc in docs:
        for topic in doc.metadata["topics"].split(", "):
            if topic:
                all_topics.add(topic)

    return RAGResponse(
        answer=answer_text,
        relevant_papers=relevant_papers,
        topics=sorted(all_topics),
    )

print("ask_researchmate() ready.")

ask_researchmate() ready.


## Cell 5 — Test the structured output

Same 3 questions as Stage 8, but now printed as clearly separated Answer / Relevant Papers / Topics sections — exactly what Stage 11's Streamlit app will display.

In [5]:
test_questions = [
    "How is Bayesian inference used in statistics?",
    "What research has been done on black holes and gravitational waves?",
    "What research has been done on transformer efficiency?",
]

for q in test_questions:
    response = ask_researchmate(q)
    print("QUESTION:", q)
    print("\nANSWER:\n", response.answer)
    print("\nRELEVANT PAPERS:")
    for title in response.relevant_papers:
        print(" -", title)
    print("\nTOPICS:", ", ".join(response.topics))
    print("=" * 70)

QUESTION: How is Bayesian inference used in statistics?

ANSWER:
 Bayesian inference is used in statistics as a general framework for updating beliefs about unknown quantities in light of data.  In the notes **“Bayesian Methods in Cosmology”** the approach is described as follows:

* Bayes’ theorem is used as the inferential engine, combining a prior distribution with the likelihood of the observed data to obtain a posterior distribution.  
* The posterior is then used to compute point estimates (e.g., posterior means or modes) and interval estimates (credible intervals).  
* Numerical sampling methods such as Markov Chain Monte Carlo (MCMC) and Nested Sampling are employed to generate samples from the posterior when it cannot be obtained analytically.  
* Bayesian model selection is performed by comparing marginal likelihoods (or Bayes factors) to assess the relative performance of competing models, contrasting this with classical p‑value approaches.

The paper **“The Frechet distribu

## What to check after running this notebook

- **Cell 2:** pipeline rebuilds without errors.
- **Cell 4:** `ask_researchmate()` runs successfully.
- **Cell 5:** for each question, confirm:
  - `answer` reads the same as Stage 8's results (same LLM, same prompt).
  - `relevant_papers` lists exactly the 3 titles the retriever found (no hallucinated titles).
  - `topics` shows a clean, deduplicated list (e.g. for the Bayesian question, expect something like `Physics, Statistics`).

Paste back the structured output for at least the Bayesian inference question — then we'll move to Stage 10 (a pleasant multi-question chat experience directly in Colab, before we build Streamlit).